A notebook to process LLM outputs into familiar formats.

## Data Loading

In [1]:

from dataset_processing import process_llm_jsonl_results, CLIRENER_LABELS_V1, transform_to_ner_format
from EXPERIMENTS.evaluate_gold import load_and_merge_gold_data
import wandb
from EXPERIMENTS.evaluate import run_nervaluate, log_to_wandb

c:\Users\Desktop\miniforge3\envs\clirener_finetune\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Based on [`run_campaign()`](EXPERIMENTS/evaluate_gold.py):

In [ ]:
# True data loading
GOLD_DATASET_ID = "P0L3/CliReNER_v_1_1_28_GOLD"
gold_data, bio_label_list = load_and_merge_gold_data(GOLD_DATASET_ID)
target_labels = list(CLIRENER_LABELS_V1)

# Predictions loading
llm_predictions_dir = "RESULTS/LLM_PREDICTIONS/ner_results_gemma_4_31b_it.jsonl" 
#"RESULTS/LLM_PREDICTIONS/ner_results_gpt_5_2_pro.jsonl"
# "RESULTS/LLM_PREDICTIONS/ner_results_gpt_5_2_pro.jsonl"
# "RESULTS/LLM_PREDICTIONS/ner_results_gemini_3_pro_preview.jsonl"
llm_name = llm_predictions_dir.split("/")[-1].replace("ner_results_", "").replace(".jsonl", "")
raw_predictions = process_llm_jsonl_results(llm_predictions_dir)
print(llm_name)

# WANDB initialization
WANDB_PROJECT = "CLIRENER_GOLD_SEEDS"

# B. Initialize WandB Run
run_name = f"eval_GOLD_{llm_name}_zs"

# Start a fresh run for this evaluation
run = wandb.init(
    project=WANDB_PROJECT,
    name=run_name,
    reinit=True, # Allow multiple runs in one script
    config={
        "model_type": "LLM_"+llm_name,
        "model_id": llm_name,
        "training_dataset": "None",
        "evaluation_dataset": GOLD_DATASET_ID, # Explicitly logging this
        "seed": -1,
        "evaluation_scope": "ALL_SPLITS_MERGED"
    }
)

# D. Transform Predictions
print("--- Transforming Predictions ---")
model_predictions_transformed = transform_to_ner_format(raw_predictions, target_labels)

pred_lookup = {}
# Iterate through all predictions (including duplicates/failed attempts in the JSONL)
for row in model_predictions_transformed[0]:
    # print(row)
    text = row['text']
    tags = row['ner_tags']
    
    # Check if tags are valid (not None) and not empty.
    # If a sentence was attempted twice (once failed, once success), 
    # this logic ensures we only store the successful one.
    if tags is not None and len(tags) > 0:
        pred_lookup[text] = tags

true_ids = []
pred_ids = []
missing_count = 0
mismatch_count = 0

for row in gold_data["test"]:
    text_key = row['text']
    
    # 3. Match by exact text
    if text_key in pred_lookup:
        p_tags = pred_lookup[text_key]
        
        # Additional safety: Ensure lengths match (LLM might have truncated text)
        if len(row['ner_tags']) == len(p_tags):
            true_ids.append(row['ner_tags'])
            pred_ids.append(p_tags)
        else:
            mismatch_count += 1
    else:
        missing_count += 1

if missing_count > 0:
    print(f"Warning: {missing_count} rows from Gold Data were not found in Predictions.")
if mismatch_count > 0:
    print(f"Warning: {mismatch_count} rows were ignored due to token length mismatch.")
    
print(f"aligned {len(true_ids)} rows for evaluation.")



# E. Calculate Metrics
# We pass the dataset's BIO scheme for ID mapping, and the target labels for reporting
results, results_by_tag = run_nervaluate(true_ids, pred_ids, bio_label_list, tags=target_labels)

# F. Log to WandB
log_to_wandb(results, results_by_tag)

print(f"SUCCESS: {run_name}")
# print(results)
wandb.finish()

--- Loading and Merging GOLD Dataset: P0L3/CliReNER_v_1_1_28_GOLD ---


Generating test split: 100%|██████████| 21/21 [00:00<00:00, 10499.51 examples/s]


Merged ['train', 'validation', 'test'] into a single dataset of size: 192
gemma_4_31b_it


wandb: Currently logged in as: andrija-2 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


--- Transforming Predictions ---
aligned 182 rows for evaluation.
--- Calculating Metrics ---


wandb: ERROR The nbformat package was not found. It is required to save notebook history.



--- Results Logged to WandB ---
              ent_type      partial       strict        exact
correct    1436.000000  1430.000000  1130.000000  1430.000000
incorrect   534.000000     0.000000   840.000000   540.000000
partial       0.000000   540.000000     0.000000     0.000000
missed      284.000000   284.000000   284.000000   284.000000
spurious     46.000000    46.000000    46.000000    46.000000
possible   2254.000000  2254.000000  2254.000000  2254.000000
actual     2016.000000  2016.000000  2016.000000  2016.000000
precision     0.712302     0.843254     0.560516     0.709325
recall        0.637090     0.754215     0.501331     0.634428
f1            0.672600     0.796253     0.529274     0.669789
SUCCESS: eval_GOLD_gemma_4_31b_it_zs


overall/exact_f1,▁
overall/exact_precision,▁
overall/exact_recall,▁
overall/partial_f1,▁
overall/partial_precision,▁
overall/partial_recall,▁
overall/strict_f1,▁
overall/strict_precision,▁
overall/strict_recall,▁
overall/type_f1,▁
+2,...


: 